## Data Exploration

In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import plotly.express as px
import matplotlib.pyplot as plt
import scipy.stats as sp

In [3]:
import os
os.chdir(r"C:\Users\belle\OneDrive\Personal Projects\athlete_profiles\athlete_clustering")

In [4]:
df = pd.read_csv(r"data\Longitudinal_Athlete_Sports_Performance_RL_Dataset.csv")  

Explore data.

In [5]:
df.head(3).T

,0,1,2
Athlete_ID,1,1,1
Session_ID,1,2,3
Days_From_Start,0,2,4
Sport,Athletics,Swimming,Swimming
Previous_Performance,64.301043,59.949164,56.71246
Previous_Training_Load,462.694009,462.694009,432.009171
Resting_Heart_Rate_bpm,65.852008,67.917641,64.709823
Heart_Rate_Variability_ms,54.573129,47.701078,68.285953
Oxygen_Saturation_pct,97.245654,98.350685,97.342644
Body_Temperature_C,36.566646,36.623072,36.582591


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Athlete_ID                   10000 non-null  int64  
 1   Session_ID                   10000 non-null  int64  
 2   Days_From_Start              10000 non-null  int64  
 3   Sport                        10000 non-null  str    
 4   Previous_Performance         10000 non-null  float64
 5   Previous_Training_Load       10000 non-null  float64
 6   Resting_Heart_Rate_bpm       10000 non-null  float64
 7   Heart_Rate_Variability_ms    9900 non-null   float64
 8   Oxygen_Saturation_pct        10000 non-null  float64
 9   Body_Temperature_C           10000 non-null  float64
 10  Sleep_Duration_hr            10000 non-null  float64
 11  Sleep_Quality_Score          9900 non-null   float64
 12  Rest_Duration_hr             10000 non-null  float64
 13  Training_Load               

In [7]:
df.describe()

,Athlete_ID,Session_ID,Days_From_Start,Previous_Performance,Previous_Training_Load,Resting_Heart_Rate_bpm,Heart_Rate_Variability_ms,Oxygen_Saturation_pct,Body_Temperature_C,Sleep_Duration_hr,...,Training_Action_ID,Training_Intensity_pct,Training_Duration_min,Training_Volume,Next_Performance,Next_Fatigue_Score,Next_Recovery_Score,Performance_Gain,Training_Efficiency,Multi_Objective_Reward
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,9900.000000,10000.000000,10000.000000,10000.000000,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,125.500000,20.500000,39.000000,68.801201,454.012940,65.282241,61.980308,97.170984,36.700628,7.203404,...,2.047100,57.228997,88.021688,51.095933,55.437999,53.376132,59.321974,4.282937,0.088428,2.338819
std,72.171815,11.543974,23.087947,8.783047,109.745419,3.882605,5.640111,0.803723,0.195794,0.900977,...,1.358406,9.422707,13.928524,14.804109,7.579011,9.677702,8.294836,3.119522,0.071695,2.068615
min,1.000000,1.000000,0.000000,32.438981,100.000000,48.859919,39.322333,94.420629,36.100000,4.500000,...,0.000000,20.951963,39.352630,11.002526,26.967617,20.320169,27.623254,-8.271482,-0.355060,-7.483354
25%,63.000000,10.750000,19.500000,62.782876,379.621009,62.657939,58.181757,96.627472,36.567152,6.591288,...,1.000000,50.884309,78.592501,40.453693,50.289450,46.950252,53.679256,2.197144,0.042861,0.964321
50%,125.500000,20.500000,39.000000,68.925588,453.863907,65.293555,61.944094,97.162011,36.698276,7.202063,...,2.000000,57.245179,88.065883,49.987970,55.468375,53.346538,59.419252,4.281885,0.083805,2.375097
75%,188.000000,30.250000,58.500000,74.855625,527.210960,67.859634,65.750753,97.716513,36.833364,7.818613,...,3.000000,63.708636,97.528082,60.359811,60.551476,59.959262,64.903274,6.392879,0.129432,3.738758
max,250.000000,40.000000,78.000000,98.547643,841.199906,81.787685,84.843426,100.000000,37.380135,9.500000,...,4.000000,90.178996,150.000000,128.564656,82.087935,91.569899,94.012243,16.736087,0.492888,10.186557


Summary stats.

Traits of clusters we might be interested in: 

* Overtraining
* Underperforming
* High natural ability
* Weakness in [speed/strength/endurance/recovery]
* Improving or not

Features to include to capture this:

* Average resting heart rate
* Average speed
* Average strength
* Average endurance
* Change in resting heart rate
* Average perfomance - training load ratio
* Average training load
* Average recovery - training load ratio
* Average performance - recovery ratio
* Change in performance
* Average change in performance


In [8]:
df.groupby(['Athlete_ID','Sport']).agg({
    'Speed_Score':'mean',
    'Strength_Score':'mean',
    'Endurance_Score':'mean'}
)

Speed_Score  Strength_Score  Endurance_Score
Athlete_ID Sport                                                   
1          Athletics     46.537391       46.983448        42.697443
           Basketball    46.057051       48.333280        46.410031
           Football      39.986306       44.524516        44.346501
           Swimming      46.426450       46.993244        46.555916
           Tennis        45.468749       49.417919        45.512440
...                            ...             ...              ...
250        Athletics     50.640398       53.255000        54.286892
           Basketball    53.875209       57.075744        51.736767
           Football      52.087057       55.004403        50.406124
           Swimming      50.272992       58.629927        47.104242
           Tennis        50.221287       53.924966        47.572806

[1250 rows x 3 columns]

In [9]:
df = df.assign(
    Recovery_TL = df['Recovery_Score'] * df['Training_Load'],
    HR_Change = df['Resting_Heart_Rate_bpm'].diff()
)

In [21]:
summary = df.groupby(['Athlete_ID']).agg({
    'Athlete_ID': 'count',
    'Sport': 'nunique',
    'Resting_Heart_Rate_bpm': 'mean',
 #   'Speed_Score': 'mean',
 #   'Strength_Score': 'mean',
    'Training_Load': 'mean',
 #   'Endurance_Score': 'mean',
    'Training_Frequency_per_Week': 'mean',
    'Training_Intensity_pct': 'mean',
    'Performance_Gain': 'mean',
    'Recovery_Score': 'mean',
    'HR_Change':'mean'
})

In [22]:
df.Sport.unique()

<StringArray>
['Athletics', 'Swimming', 'Basketball', 'Tennis', 'Football']
Length: 5, dtype: str

In [23]:
summary = summary.assign(
    Swim_Performance= df.query('Sport == "Swimming"').groupby('Athlete_ID')['Current_Performance'].mean(),
    Athletics_Performance= df.query('Sport == "Athletics"').groupby('Athlete_ID')['Current_Performance'].mean(),
    Basketball_Performance= df.query('Sport == "Basketball"').groupby('Athlete_ID')['Current_Performance'].mean(),
    Tennis_Performance= df.query('Sport == "Tennis"').groupby('Athlete_ID')['Current_Performance'].mean(),
    Football_Performance= df.query('Sport == "Football"').groupby('Athlete_ID')['Current_Performance'].mean()
    )
summary

,Athlete_ID,Sport,Resting_Heart_Rate_bpm,Training_Load,Training_Frequency_per_Week,Training_Intensity_pct,Performance_Gain,Recovery_Score,HR_Change,Swim_Performance,Athletics_Performance,Basketball_Performance,Tennis_Performance,Football_Performance
Athlete_ID,,,,,,,,,,,,,,
1,40,5,65.772410,464.131621,4.987120,57.386944,3.727136,61.447361,0.007320,46.585309,46.427470,45.451501,46.886513,45.353476
2,40,5,65.206532,451.451094,4.943073,58.334546,4.134232,61.702628,-0.007319,49.641526,46.436048,53.801255,47.636215,50.271827
3,40,5,65.392055,439.117259,5.017330,57.451863,3.995008,59.137553,-0.039756,50.274125,51.229564,53.905913,53.738263,49.815643
4,40,5,65.049863,453.578849,5.022981,59.667260,4.786080,62.303687,0.079593,50.696158,48.446249,51.173479,49.613254,48.936902
5,40,5,65.904176,431.143973,5.133898,57.676157,3.975167,61.310048,-0.039560,49.112421,51.526629,48.136518,47.446515,50.132827
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,40,5,64.898869,450.447899,5.156643,55.832042,3.756954,59.957665,0.097964,47.644778,47.018620,49.541184,48.816981,45.777347
247,40,5,65.752937,435.636603,4.652908,56.560654,3.940373,60.663118,0.040396,57.394910,58.293907,62.103005,61.274161,57.857463
248,40,5,64.632669,444.076951,4.976780,57.259784,3.984451,61.494494,-0.127916,51.158315,53.740414,55.911792,52.155503,55.083135


In [30]:
fig = px.scatter_matrix(summary, 
                  dimensions=['Resting_Heart_Rate_bpm', 
                              #'Speed_Score', 
                              #'Strength_Score', 
                              'Training_Load', 
                              'Training_Intensity_pct',
                              #'Endurance_Score', 
                              'Training_Frequency_per_Week', 
                              'Performance_Gain',
                              'Recovery_Score',
                              'HR_Change',
                                'Swim_Performance',
                                'Athletics_Performance',
                                'Basketball_Performance',
                                'Tennis_Performance',
                                'Football_Performance'
                              ], 
                  title='Scatter Matrix of Athlete Summary Metrics by Sport',
                  height=1200, width=1200, opacity=0.7, )
fig.update_layout(font=dict(size=6), title_font=dict(size=10))

fig.show()

In [31]:
summary.to_csv(r"data\athlete_summary.csv", index=False)